# Method 3 - k-means + Canonical Autoencoder

Non-linear dimensionality reduction with a single autoencoder (219 -> 6 -> 4 ->
2, SELU) followed by k-means on the 2-D latent space.

**What changed vs the original**
- Shared `src/` preprocessing; the AE + training loop live in `src.autoencoders`
  and are now **seeded/deterministic** (the original never seeded torch).
- Shared majority filter + `finalize_method`; intrinsic metrics on the 2-D latent.

In [ ]:
# Make the shared src/ package importable regardless of CWD.
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.seeding import seed_everything
from src.preprocessing import load_and_preprocess_enmap
from src.results import finalize_method
from src.postprocess import majority_filter

seed_everything()          # seed numpy / random / torch from config.SEED
print(f"Seed = {config.SEED}")

In [ ]:
from sklearn.cluster import KMeans
from src.autoencoders import Autoencoder, train_autoencoder, encode_full, get_device

# Autoencoders reconstruct reflectance in [0,1] (Sigmoid out) -> keep unstandardised.
res = load_and_preprocess_enmap(config.ENMAP_TIF, standardize=False)
X = res.data_valid
device = get_device(); print("device:", device)

ae = Autoencoder(X.shape[1], encoder_dims=(6, 4, 2), l1_lambda=1e-4)
history = train_autoencoder(ae, X, epochs=20, batch_size=64, learning_rate=1e-4,
                            seed=config.SEED, device=device, name="Canonical AE")
latent = encode_full(ae, X, device)
print("latent:", latent.shape)

plt.figure(figsize=(8,4)); plt.plot(history["loss"]); plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("Canonical AE training"); plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "ca_training.png", dpi=200); plt.show()

In [ ]:
km = KMeans(n_clusters=config.KMEANS_K, random_state=config.SEED, n_init=config.KMEANS_N_INIT)
labels_raw = km.fit_predict(latent)

ca_map = finalize_method("ca", res, labels_raw, features_for_intrinsic=latent)

def plot_cluster_map(labels_spatial, extent, title):
    n = int(np.nanmax(labels_spatial)) + 1 if np.isfinite(np.nanmax(labels_spatial)) else 1
    fig, ax = plt.subplots(figsize=(12, 9))
    cmap = plt.get_cmap("rainbow", n).copy()
    cmap.set_bad(color="white")
    im = ax.imshow(labels_spatial, cmap=cmap, extent=extent, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax, label="Cluster", fraction=0.046)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    return fig, ax

plot_cluster_map(ca_map, res.extent, f"k-means + Canonical AE (k={config.KMEANS_K})")
plt.savefig(config.FIGURES_DIR / "ca_classification.png", dpi=200, bbox_inches="tight"); plt.show()